In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("schema_prefix", "elina_sharabura")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA_PREFIX = dbutils.widgets.get("schema_prefix")

SILVER_CUSTOMERS = f"{CATALOG}.{SCHEMA_PREFIX}_silver.customers"
SILVER_PRODUCTS = f"{CATALOG}.{SCHEMA_PREFIX}_silver.products"
DIM_CUSTOMER = f"{CATALOG}.{SCHEMA_PREFIX}_gold.dim_customer"
DIM_PRODUCT = f"{CATALOG}.{SCHEMA_PREFIX}_gold.dim_product"

SCD2_BEGINNING_OF_TIME = "1900-01-01 00:00:00"
SCD2_END_OF_TIME = "9999-12-31 23:59:59"
UNKNOWN_MEMBER = "Unknown"
SILVER_ORDERS = f"{CATALOG}.{SCHEMA_PREFIX}_silver.orders"
DIM_COUNTRY = f"{CATALOG}.{SCHEMA_PREFIX}_gold.dim_country"
DIM_ORDER_CHANNEL = f"{CATALOG}.{SCHEMA_PREFIX}_gold.dim_order_channel"

In [0]:
spark.sql(f"""
INSERT OVERWRITE {DIM_CUSTOMER}
    (customer_sk, customer_id, customer_name, email, country, age, loyalty_tier,
     signup_date, marketing_opt_in, effective_from, effective_to, is_current)
SELECT
    customer_sk,
    customer_id,
    name,
    email,
    country,
    age,
    loyalty_tier,
    signup_date,
    marketing_opt_in,
    CASE
        WHEN effective_from = MIN(effective_from) OVER (PARTITION BY customer_id)
        THEN TIMESTAMP '{SCD2_BEGINNING_OF_TIME}'
        ELSE effective_from
    END,
    COALESCE(effective_to, TIMESTAMP '{SCD2_END_OF_TIME}'),
    is_current
FROM {SILVER_CUSTOMERS}
""")

In [0]:
spark.sql(f"""
INSERT OVERWRITE {DIM_PRODUCT}
    (product_id, product_name, category, price_usd)
SELECT
    product_id,
    name,
    COALESCE(category, '{UNKNOWN_MEMBER}'),
    CAST(price_usd AS DECIMAL(12,2))
FROM {SILVER_PRODUCTS}
""")

In [0]:
spark.sql(f"""
MERGE INTO {DIM_COUNTRY} t
USING (
    SELECT DISTINCT COALESCE(country, '{UNKNOWN_MEMBER}') AS country_code
    FROM {SILVER_ORDERS}
) s
ON t.country_code = s.country_code
WHEN NOT MATCHED THEN INSERT (country_code) VALUES (s.country_code)
""")

In [0]:
spark.sql(f"""
MERGE INTO {DIM_ORDER_CHANNEL} t
USING (
    SELECT DISTINCT
        COALESCE(payment_method, '{UNKNOWN_MEMBER}') AS payment_method,
        COALESCE(device,         '{UNKNOWN_MEMBER}') AS device,
        COALESCE(source,         '{UNKNOWN_MEMBER}') AS source
    FROM {SILVER_ORDERS}
) s
ON  t.payment_method = s.payment_method
AND t.device         = s.device
AND t.source         = s.source
WHEN NOT MATCHED THEN INSERT (payment_method, device, source)
VALUES (s.payment_method, s.device, s.source)
""")